# Project Thesis: Using Taproot for Document Verification


In [1]:
# Librerias usadas

import hashlib
from bitcoinutils.setup import setup
from bitcoinutils.keys import PrivateKey, P2trAddress
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.script import Script
from coincurve import PrivateKey as ccPrivateKey , PublicKeyXOnly 
setup("regtest")


'regtest'

# Sign Documents 

Create a Issuer pair Schnorr Key (skiss,pkiss). 

In [6]:
# Issuer Key

seed = hashlib.sha256(b"Dean").digest()
sk = ccPrivateKey(seed)

index1 = int(1).to_bytes(4, "big").hex()
message1 = b"mensaje de prueba 1"
hash_doc1 = hashlib.sha256(message1).digest()
sig1 = sk.sign_schnorr(hash_doc1)
pk_xonly_bytes = sk.public_key_xonly.format()
pk_xonly = PublicKeyXOnly(pk_xonly_bytes)

CONTROL1 = [index1,sig1,pk_xonly,hash_doc1]

index2 = int(2).to_bytes(4, "big").hex()
message2 = b"mensaje de prueba 2"
hash_doc2 = hashlib.sha256(message2).digest()
sig2 = sk.sign_schnorr(hash_doc2)
CONTROL2 = [index2,sig2,pk_xonly,hash_doc2]

index3 = int(3).to_bytes(4, "big").hex()
message3 = b"mensaje de prueba 3"
hash_doc3 = hashlib.sha256(message3).digest()
sig3 = sk.sign_schnorr(hash_doc3)
CONTROL3 = [index3,sig3,pk_xonly,hash_doc3]

index4 = int(4).to_bytes(4, "big").hex()
message4 = b"mensaje de prueba 4"
hash_doc4 = hashlib.sha256(message4).digest()
sig4 = sk.sign_schnorr(hash_doc4)
CONTROL4 = [index4,sig4,pk_xonly,hash_doc4]


In [7]:
# Verification 
CONTROL[2].verify(CONTROL[1], CONTROL[3])

NameError: name 'CONTROL' is not defined

## Taproot Script Creation

In [8]:
btc_sk = PrivateKey(secret_exponent=1)
btc_pub = btc_sk.get_public_key()
btc_xonly = btc_pub.to_x_only_hex()

In [9]:

leaf1 = Script([CONTROL1[0], CONTROL1[3].hex(), CONTROL1[2].format().hex(), "OP_FALSE"])
leaf2 = Script([CONTROL2[0], CONTROL2[3].hex(), CONTROL2[2].format().hex(), "OP_FALSE"])
leaf3 = Script([CONTROL3[0], CONTROL3[3].hex(), CONTROL3[2].format().hex(), "OP_FALSE"])
leaf4 = Script([CONTROL4[0], CONTROL4[3].hex(), CONTROL4[2].format().hex(), "OP_FALSE"])

leafs = [
    [leaf1,leaf2],
    [leaf3,leaf4]
]

taproot_addr = btc_pub.get_taproot_address([leafs])
print(taproot_addr.to_string())


bcrt1p8kzmft58a90ptaxr8kapmjm90gnvzx57mrks0g0rgk0vxgs4smdqrfnxks


In [10]:
from bitcoinutils.utils import tapleaf_tagged_hash, tapbranch_tagged_hash

h1 = tapleaf_tagged_hash(leaf1)
h2 = tapleaf_tagged_hash(leaf2)
h3 = tapleaf_tagged_hash(leaf3)
h4 = tapleaf_tagged_hash(leaf4)

print(h1.hex())
print(h2.hex())
print(h3.hex())
print(h4.hex())

4303fa2bca673746a2ebbeed26861e801454e7e60132bedcc0d979fea6fb244b
f5c5d5823b4cbf6951d0ea0e09f52d8a45b12f70e1887824c564477fb1222b19
e608efe72ff862e6e86a76f5fecbcceb5b53d3fe4212002f2bfc7b68de46ed53
adc40d1bcd7e300b7ada1640601ecfa84181b71e77c2c39100c0c9e46b4c75db


# Reconstruction and Verification of a document

Take one script leaf to verify, in this example we take message3

In [11]:
b12 = tapbranch_tagged_hash(h1, h2)
CONTROL3 = [index3,sig3,pk_xonly,hash_doc3]
PROOF = [b12,h4]

leaf3_rebuilt = Script([
    CONTROL3[0],                  # index3 hex, 4 bytes
    CONTROL3[3].hex(),            # hash_doc3 -> hex
    CONTROL3[2].format().hex(),   # pk_xonly -> xonly hex
    "OP_FALSE",
])


h3_rebuilt = tapleaf_tagged_hash(leaf3_rebuilt)
b12 = PROOF[0]
h4  = PROOF[1]
b34_rebuilt = tapbranch_tagged_hash(h3_rebuilt, h4)
root_rebuilt = tapbranch_tagged_hash(b12, b34_rebuilt)

print("h3:", h3_rebuilt.hex())
print("b34:", b34_rebuilt.hex())
print("root:", root_rebuilt.hex())

taproot_addr_rebuilt = btc_pub.get_taproot_address(root_rebuilt)
print(taproot_addr_rebuilt.to_string())
print('bcrt1p8kzmft58a90ptaxr8kapmjm90gnvzx57mrks0g0rgk0vxgs4smdqrfnxks ')

h3: e608efe72ff862e6e86a76f5fecbcceb5b53d3fe4212002f2bfc7b68de46ed53
b34: e43d805ac6265c44d7b5e08d70512041d70a738a108382c9b9da067953b306be
root: 8b40cbf341790a9fd05b04799c150a599413d2e273a37afb3ac527a8981dccdd
bcrt1p8kzmft58a90ptaxr8kapmjm90gnvzx57mrks0g0rgk0vxgs4smdqrfnxks
bcrt1p8kzmft58a90ptaxr8kapmjm90gnvzx57mrks0g0rgk0vxgs4smdqrfnxks 


In [12]:
taproot_addr_rebuilt.to_string() == taproot_addr.to_string()

True

In [ ]:
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.script import Script
from bitcoinutils.keys import P2wpkhAddress

# UTXO a gastar
txid_in = "7f27bf0ce292863c3f095cee282924d24c980575e54035ea0fdc38d9741fe491"
vout_in = 1
amount_in = 1 * 100_000_000  # ejemplo

# destino normal
addr_out = "bcrt1qdgljja33z3gxmvjw95gy6uvndrf5hfj6qle3zu"
addr_out_obj = P2wpkhAddress(addr_out)
amount_out = 50_000_000

# dato OP_RETURN
opret_data = b"Con esto acabo la tesis ejemplo"
opret_script = Script(["OP_RETURN", opret_data.hex()])
opret_out = TxOutput(0, opret_script)

# fee y cambio
fee = 1000
change = amount_in - amount_out - fee

# input
txin = TxInput(txid_in, vout_in)

# outputs
out1 = TxOutput(amount_out, addr_out_obj.to_script_pub_key())
out2 = TxOutput(change, taproot_addr.to_script_pub_key())

# tx
tx = Transaction(
    [txin],
    [out1, opret_out, out2],
    has_segwit=True
)

# scriptPubKey y amount del UTXO que estás gastando
utxo_scripts = [taproot_addr.to_script_pub_key()]
amounts = [amount_in]

# firma key-path
sig = btc_sk.sign_taproot_input(
    tx,
    0,
    utxo_scripts,
    amounts,
    script_path=False,
    tapleaf_scripts=leafs,
)

tx.witnesses.append(TxWitnessInput([sig]))

rawtx = tx.serialize()
print(rawtx)
print("TXID:", tx.get_txid())

TypeError: Amount needs to be in satoshis as an integer

In [ ]:
print(bytes.fromhex("436f6e206573746f20616361626f206c6120746573697320656a656d706c6f").decode('utf-8'))

Con esto acabo la tesis ejemplo
